# Generate reusable matched MVTec, VisA, or combined splits

Set `DATASETS = ('mvtec', 'visa', 'both')` to generate every manifest in one run, or use a one-item tuple for a single collection. The notebook creates deterministic, category-balanced and label-balanced 50/50 manifests that can be reused directly by the adversarial benchmark. The `'both'` artifact is also used for `mvtec_to_visa` and `visa_to_mvtec` transfer experiments.

In [ ]:
import subprocess
import sys
from pathlib import Path

print('===== STEP 1: CLONE/UPDATE REPOSITORY =====')
working = Path('/kaggle/working')
repo_root = working / 'adversarial-robustness'
repo_url = 'https://github.com/Parsagh05/adversarial-robustness.git'
if repo_root.exists():
    subprocess.run(['git', '-C', str(repo_root), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', repo_url, str(repo_root)], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(repo_root / 'requirements.txt')],
    check=True,
)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print('Repository ready:', repo_root)

In [ ]:
import json
from adversarial_harness import create_matched_split_manifest

print('===== STEP 2: GENERATE MATCHED SPLITS =====')
# Generate all three in one run. Use ('mvtec',), ('visa',), or ('both',)
# when only one artifact is needed. Keep the trailing comma for one item.
DATASETS = ('mvtec', 'visa', 'both')
MVTEC_ROOT = Path('/kaggle/input/datasets/alirezasalehy/mvtec-ad/mvtec_anomaly_detection')
VISA_ROOT = Path('/kaggle/input/datasets/alirezasalehy/visa-ad/VisA_20220922')
SPLIT_SEED = 111
FIT_FRACTION = 0.5
CATEGORIES_BY_DATASET = {
    'mvtec': None,  # None means every category in that collection.
    'visa': None,
    'both': None,
}

VALID_DATASETS = {'mvtec', 'visa', 'both'}
if not DATASETS:
    raise ValueError('DATASETS cannot be empty')
unknown = sorted(set(DATASETS) - VALID_DATASETS)
if unknown:
    raise ValueError(f'Unknown DATASETS values: {unknown}')
if len(set(DATASETS)) != len(DATASETS):
    raise ValueError(f'DATASETS contains duplicates: {DATASETS}')
if set(DATASETS) & {'mvtec', 'both'} and not MVTEC_ROOT.is_dir():
    raise FileNotFoundError(f'MVTec mount not found: {MVTEC_ROOT}')
if set(DATASETS) & {'visa', 'both'} and not VISA_ROOT.is_dir():
    raise FileNotFoundError(f'VisA mount not found: {VISA_ROOT}')

split_dir = Path('/kaggle/working/splits')
generated_artifacts = []
for dataset in DATASETS:
    artifact_name = f'{dataset}_matched_test_per_category_v1_seed{SPLIT_SEED}'
    csv_path = split_dir / f'{artifact_name}.csv'
    json_path = split_dir / f'{artifact_name}.json'
    create_matched_split_manifest(
        dataset=dataset,
        mvtec_root=str(MVTEC_ROOT) if dataset in {'mvtec', 'both'} else None,
        visa_root=str(VISA_ROOT) if dataset in {'visa', 'both'} else None,
        csv_path=str(csv_path),
        json_path=str(json_path),
        categories=CATEGORIES_BY_DATASET[dataset],
        split_seed=SPLIT_SEED,
        fit_fraction=FIT_FRACTION,
    )
    metadata = json.loads(json_path.read_text(encoding='utf-8'))
    generated_artifacts.append((csv_path, json_path))
    print(f'\n[{dataset}]')
    print('CSV:', csv_path)
    print('JSON:', json_path)
    print('CSV SHA256:', metadata['csv_sha256'])
    print('Selected rows:', metadata['total_selected_rows'])
    for category, counts in metadata['category_counts'].items():
        print(category, counts)

print(f'Generated {len(generated_artifacts)} matched manifest pair(s).')

In [ ]:
import zipfile

print('===== STEP 3: PACKAGE OUTPUTS =====')
archive = Path('/kaggle/working') / f'dataset_matched_splits_seed{SPLIT_SEED}.zip'
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for csv_path, json_path in generated_artifacts:
        bundle.write(csv_path, arcname=f'splits/{csv_path.name}')
        bundle.write(json_path, arcname=f'splits/{json_path.name}')
print('Packaged all generated split manifests:', archive)